# Entrenamiento del modelo de lenguaje del asistente

El asistente necesita **entender lo que escribe el operador**: ¿qué intención tiene la pregunta (qué hago, por qué sube el riesgo, cuánto tiempo tengo…) y qué falla menciona (calor, potencia, torque, desgaste)? Aquí se entrena ese modelo con datos, se elige su configuración con validación y se mide con preguntas que no vio.

**Qué es y qué no es.** Es un clasificador de texto entrenado en este proyecto (TF-IDF + regresión logística). No genera texto ni es un modelo de lenguaje grande. Las cifras de riesgo de las respuestas no salen de aquí: las calcula el pronosticador de fallas (notebook de la planta y `src/forecast.py`).

**Aviso sobre los datos.** No existe un corpus real de operadores de fresadoras. Las preguntas de entrenamiento se generaron combinando frases escritas a mano con variaciones (cortesías, tildes, errores de tecleo). Para no engañarnos, la partición se hace por *familias de frases*: las frases de validación y prueba nunca aparecen en entrenamiento. Además hay un conjunto **natural** de 69 preguntas escritas aparte, a mano.

In [1]:
from pathlib import Path
import sys, importlib
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import assistant_data as D
T = importlib.import_module("scripts.train_assistant")
data, nat = D.build(seed=0), D.natural()
print({k: len(v) for k, v in data.items()}, "| natural:", len(nat))
print("Intenciones:", D.INTENTS)
print("Fallas:", D.CAUSES)

{'train': 4437, 'val': 700, 'test': 723} | natural: 69
Intenciones: ['que_hago', 'por_que', 'cuanto', 'seguir', 'prevenir', 'planta', 'fuera']
Fallas: ['ninguna', 'HDF', 'PWF_high', 'PWF_low', 'OSF', 'TWF']


## Ejemplos del conjunto de datos

Cada fila es (pregunta, intención, falla mencionada). Se ven las variaciones: sin tildes, con errores de tecleo, con número de máquina.

In [2]:
for text, intent, cause in data["train"][:8]:
    print(f"{text!r:62} -> {intent:9} {cause}")
print()
for text, intent, cause in nat[:5]:
    print("NATURAL:", f"{text!r:62} -> {intent:9} {cause}")

'Disuclpa, sigo con el lote aquí'                              -> seguir    ninguna
'qué tan pornto falla con la potencia insuficiente ahora en la 4' -> cuanto    PWF_low
'una consulta: a qué atribuyo que se calenta mucho hoy de fresadora 01!' -> por_que   HDF
'oye, necesito resolver la carga mecanica excesiva e esta maquina?' -> que_hago  OSF
'mira, cómo lo arreglo ahora'                                  -> que_hago  ninguna
'A que atribuyo el esfuerzo excesivo en la herramienta gracias' -> por_que   OSF
'Disclpa, qué causa que casi no corta aquí en la fresadora número 4...' -> por_que   PWF_low
'Urgene: con la potencia que cayo, la paro o sigo porfa de fresadora 11' -> seguir    PWF_low

NATURAL: 'la fresadora 4 se está calentando demasiado, ¿qué hago?'      -> que_hago  HDF
NATURAL: '¿por qué me sale alerta en la 7?'                             -> por_que   ninguna
NATURAL: 'me quedan pocos minutos antes de que falle?'                  -> cuanto    ninguna
NATURAL: 'sigo cortando o 

## Entrenamiento y selección

Se entrenan dos cabezas (intención y falla) sobre las mismas variables de texto: n-gramas de caracteres (toleran faltas de ortografía), palabras y, en la tercera configuración, señales binarias de raíces típicas de cada falla (`calor`, `desgast`, `torque`…) cuyo peso aprende el modelo. Se prueban 3 configuraciones × 3 valores de regularización `C` y **se elige con validación**.

In [3]:
from sklearn.metrics import f1_score
Xtr, ytr_i, ytr_c = T.columns(data["train"]); Xva, yva_i, yva_c = T.columns(data["val"])
rows, best = [], None
for kind in ("char", "char+word", "char+word+palabras_clave"):
    for C in (1, 10, 100):
        mi, mc = T.make(kind, C).fit(Xtr, ytr_i), T.make(kind, C).fit(Xtr, ytr_c)
        fi, fc = f1_score(yva_i, mi.predict(Xva), average="macro"), f1_score(yva_c, mc.predict(Xva), average="macro")
        rows.append((kind, C, fi, fc)); best = best if best and best[0] >= (fi + fc) / 2 else ((fi + fc) / 2, kind, C, mi, mc)
print(f"{'variables':26} {'C':>4} {'F1 intención':>13} {'F1 falla':>9}")
for kind, C, fi, fc in rows:
    print(f"{kind:26} {C:>4} {fi:13.3f} {fc:9.3f}")
print("\nElegida:", best[1], "C =", best[2])

variables                     C  F1 intención  F1 falla
char                          1         0.764     0.683
char                         10         0.761     0.698
char                        100         0.758     0.692
char+word                     1         0.774     0.644
char+word                    10         0.771     0.676
char+word                   100         0.774     0.694
char+word+palabras_clave      1         0.792     0.916
char+word+palabras_clave     10         0.785     0.917
char+word+palabras_clave    100         0.779     0.920

Elegida: char+word+palabras_clave C = 1


Las palabras clave suben mucho la comprensión de la falla en validación (de ~0,70 a ~0,92): con pocas frases de ejemplo, darle al modelo una pista explícita ayuda más que más capacidad.

In [4]:
import joblib
from src.assistant import language_model
bundle = language_model()   # el modelo guardado y verificado (hash y versiones)
print("Umbral de rechazo:", bundle["threshold"], "| intenciones:", [str(i) for i in bundle["intents"]])

Umbral de rechazo: 0.0 | intenciones: ['cuanto', 'fuera', 'planta', 'por_que', 'prevenir', 'que_hago', 'seguir']


## Evaluación

Dos conjuntos que **no se usaron para elegir nada**: la partición de prueba (frases y marcos de pregunta nunca vistos) y el conjunto natural escrito a mano. Se compara con la versión anterior del asistente, que solo medía similitud con ejemplos, sin entrenamiento propio.

In [5]:
for name, rows_ in (("Prueba (frases no vistas)", data["test"]), ("Natural (a mano)", nat)):
    X, y_i, y_c = T.columns(rows_)
    pi, pc, _, _ = T.predict(bundle, X)
    bi, bc = T.baseline(rows_)
    m, b = T.metrics(y_i, y_c, pi, pc), T.metrics(y_i, y_c, bi, bc)
    print(name, f"({len(rows_)} preguntas)")
    print(f"  entrenado    intención {m['intent_accuracy']:.0%}   falla {m['cause_accuracy']:.0%}")
    print(f"  sin entrenar intención {b['intent_accuracy']:.0%}   falla {b['cause_accuracy']:.0%}")

Prueba (frases no vistas) (723 preguntas)
  entrenado    intención 83%   falla 86%
  sin entrenar intención 50%   falla 87%
Natural (a mano) (69 preguntas)
  entrenado    intención 80%   falla 96%
  sin entrenar intención 49%   falla 93%


## Dónde se equivoca

Un modelo honesto muestra sus errores. Estas son las preguntas naturales que clasificó mal.

In [6]:
X, y_i, y_c = T.columns(nat)
pi, pc, _, _ = T.predict(bundle, X)
errs = [(t, a, b, p, q) for (t, a, b), p, q in zip(nat, pi, pc) if (p, q) != (a, b)]
print(len(errs), "errores de", len(nat))
for t, a, b, p, q in errs[:10]:
    print(f"- {t!r}\n    esperado: {a}/{b}   predicho: {p}/{q}")

16 errores de 69
- 'el motor está jalando muchísimo, creo que se sobrecarga'
    esperado: que_hago/PWF_high   predicho: por_que/PWF_high
- 'la máquina casi no está cortando, tiene poca potencia'
    esperado: que_hago/PWF_low   predicho: seguir/PWF_low
- 'el refrigerante no está enfriando como antes'
    esperado: por_que/HDF   predicho: que_hago/HDF
- '¿se podrá terminar el pedido con esta herramienta tan vieja?'
    esperado: seguir/TWF   predicho: cuanto/TWF
- '¿qué máquina reviso primero hoy?'
    esperado: planta/ninguna   predicho: fuera/ninguna
- '¿hay riesgo de que se me rompa la fresa?'
    esperado: cuanto/TWF   predicho: por_que/TWF
- 'auxilio, la 12 está en rojo'
    esperado: que_hago/ninguna   predicho: planta/ninguna
- 'qué pasa si la dejo funcionando así'
    esperado: seguir/ninguna   predicho: por_que/ninguna
- 'está muy caliente la temperatura del proceso'
    esperado: por_que/HDF   predicho: que_hago/HDF
- '¿debo cambiar la herramienta ya?'
    esperado: que_hago/

## Conclusiones y límites

- El modelo entiende la **intención** de preguntas nuevas en ~80 % de los casos y la **falla mencionada** en ~86–96 %, frente a ~50 % de la versión sin entrenar. Cuando no entiende, el asistente lo dice y sugiere preguntas.
- **Limitación declarada:** el entrenamiento se ejecutó dos veces. Tras ver la primera evaluación (intención 65–77 %) se ampliaron las frases y se añadieron las palabras clave; por eso la prueba y el conjunto natural ya no son totalmente independientes de mis decisiones de diseño. Las cifras son una buena estimación, no una garantía.
- Los datos son sintéticos y en español de un solo autor: con operadores reales habría más jerga y más ruido.
- El texto de la respuesta sale de plantillas; solo con una clave de API opcional lo redacta Claude. Las cifras de riesgo siempre las calcula el pronosticador.